# A solar track, from the file to the flux

This notebook rebuilds the plot the Observe tab draws for a **solar track** —
flux in SFU against UTC, corrected to above the atmosphere, with the tracking
scallop taken out — starting from the recorded HDF5 and doing each step by
hand.

The point is the steps, not the picture. `observation_plot.plot_observation`
will draw the same thing in one line, and the last section shows that, but
then you cannot see what was assumed. Everything here is `h5py`, `numpy` and
`matplotlib` except the scallop, which needs two modules from the repository
for a reason explained when we get there.

**What a solar track is**: an ordinary tracked observation of the Sun. The
mount follows it, the receiver records full spectra every 3 s, and the
interesting quantity is not the spectrum — which for the Sun is a flat
continuum — but the band power through time.

Run it from anywhere in the repository; the next cell finds the rest.

## 0. Find the repository

Recordings live in `receiver_scheduler/data/observations/`, and the scallop section imports `scallop.py` from `receiver_scheduler/`. This cell locates both from wherever the notebook is being run.


In [ ]:
import json
from pathlib import Path

import h5py
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline
plt.rcParams['figure.dpi'] = 120
plt.rcParams['figure.figsize'] = (11, 4)

In [ ]:
# Where things are. The notebook finds the repository from its own location,
# so it runs from anywhere - the notebooks folder, the root, or a copy
# elsewhere with the repo path given as SRT_ROOT.
import os
import sys
from pathlib import Path

def _find_root(start=None):
    here = Path(os.environ.get('SRT_ROOT') or start or Path.cwd()).resolve()
    for d in (here, *here.parents):
        if (d / 'receiver_scheduler' / 'h1_web_scheduler.py').exists():
            return d
    raise SystemExit('cannot find the repository: set SRT_ROOT to its path')

ROOT = _find_root()
SCHED = ROOT / 'receiver_scheduler'
OBS = SCHED / 'data' / 'observations'
if str(SCHED) not in sys.path:
    sys.path.insert(0, str(SCHED))   # scallop, observation_plot, drift_fit, ...

print(f'repository {ROOT}')
print(f'recordings {OBS}  ({len(list(OBS.glob("*.h5")))} files)')


## 1. Pick a solar track

Recordings are `data/observations/YYYYMMDD_HHMMSS_<mode>.h5`. `track` means the
mount followed the sky; `drift` means it was parked. We want a `track` whose
`object_name` is the Sun.

In [ ]:
folder = OBS
tracks = []
for p in sorted(folder.glob('*_track.h5')):
    try:
        with h5py.File(p, 'r') as f:
            if str(f.attrs.get('object_name', '')).lower() == 'sun':
                tracks.append((p, f['timestamps'].shape[0]))
    except OSError:
        pass  # being written right now, or not readable

for i, (p, n) in enumerate(tracks):
    print(f'  [{i}] {p.name}  {n} records')

FILE_INDEX = -1          # -1 = most recent
h5_path = tracks[FILE_INDEX][0]
print(f'\nUsing {h5_path.name}')

## 2. Open it, and read the attributes that matter

A recording still being written needs `swmr=True`; a finished one does not, and
a plain open is the right first try.

Two attributes decide everything that follows:

- **`spectra_wide_units`** — `K` means the bandpass template and the gain in
  force were applied at write time *and the system temperature subtracted*, so
  the dataset is already antenna temperature. `counts` would mean they did not
  apply to that tuning. The **dataset name** carries the same fact
  (`spectra_wide_kelvin` vs `spectra_wide_linear`), which is the real guard:
  ask for the wrong one and you get a `KeyError` rather than the other scale.
- **`continuum_band_hz`** — the span this file's continuum is measured over.

In [ ]:
try:
    hf = h5py.File(h5_path, 'r')
except OSError:
    hf = h5py.File(h5_path, 'r', swmr=True)
    print('(a recording in progress: reading what has arrived so far)')

attrs = dict(hf.attrs)

for k in ('obs_name', 'observation_mode', 'object_name', 'spectra_wide_units',
          'gain_db', 'applied_t_sys_k', 'applied_gain_counts_per_k',
          'beam_fwhm_deg', 'effective_area_m2', 'nominal_integration_time'):
    print(f'{k:28s} {attrs.get(k)}')

print()
print('continuum band  ', attrs['continuum_band_hz'] / 1e6, 'MHz')
print('H I band        ', attrs['h1_band_hz'] / 1e6, 'MHz')
print('datasets        ', [k for k in hf if k.startswith('spectra') or k in
                           ('timestamps', 'frequency_hz_wide', 'pilot_burst')])

## 3. Load the continuum product and drop the calibration bursts

The receiver records **two products** from one stream: the H I sub-band in fine
channels, and the whole 8 MHz as the continuum product. A solar track is a
continuum measurement, so we want the wide one.

`pilot_burst = 1` marks a record during which the calibration transmitter sent
its comb across the band. Those records are not sky and must come out. (With
the transmitter not yet wired to the vertex dipole nothing is actually
transmitted, but the records are still flagged and still dropped, so the
reduction is the same either way.)

In [ ]:
freq = hf['frequency_hz_wide'][:]
spec = hf['spectra_wide_kelvin'][:]          # antenna temperature, K
t = hf['timestamps'][:]                      # epoch seconds, UTC
burst = hf['pilot_burst'][:].astype(bool)

print(f'{spec.shape[0]} records x {spec.shape[1]} channels')
print(f'{burst.sum()} calibration bursts dropped')

spec, t = spec[~burst], t[~burst]
print(f'{spec.shape[0]} science records, '
      f'{(t[-1] - t[0]) / 60:.1f} min from {np.datetime64(int(t[0]), "s")} UTC')

## 4. Choose the channels — the continuum window

Not every channel can go into the band mean:

- **The H I band is excluded.** We are measuring a broadband source; hydrogen
  emission along the line of sight is a contaminant here, and it varies with
  where the dish is pointed.
- **The LO spur is excluded.** The tuning puts the DC artefact at the band edge
  on purpose, where no line lives, but it is still an artefact.
- **What is left** is 1415.7–1418.8 MHz, the good side of the SAW filter, which
  by construction holds no hydrogen.

This is `drift_fit._band_window` written out.

In [ ]:
DC_MASK_HZ = 40e3        # half-width of the LO artefact mask

lo, hi = attrs['continuum_band_hz']
h1_lo, h1_hi = attrs['h1_band_hz']
dc = float(attrs.get('dc_artefact_freq_hz', attrs['center_freq_hz']))

keep = (freq >= lo) & (freq <= hi)
keep &= np.abs(freq - dc) > DC_MASK_HZ
keep &= ~((freq >= h1_lo) & (freq <= h1_hi))

print(f'{keep.sum()} of {keep.size} channels kept, '
      f'{freq[keep].min()/1e6:.3f}-{freq[keep].max()/1e6:.3f} MHz')

fig, ax = plt.subplots()
ax.plot(freq / 1e6, spec[len(spec) // 2], lw=0.7, color='0.6', label='one record')
ax.plot(freq[keep] / 1e6, spec[len(spec) // 2][keep], lw=0.9, color='C0',
        label='kept for the band mean')
ax.axvspan(h1_lo / 1e6, h1_hi / 1e6, color='C3', alpha=0.12, label='H I band (excluded)')
ax.set_xlabel('MHz'); ax.set_ylabel('Antenna temperature (K)')
ax.set_title('One record of a solar track: a flat continuum')
ax.legend(fontsize=8); plt.show()

## 5. Band power per record

The mean over those channels, one number per record. This is the raw light
curve, and it already looks like the final plot — everything after this is
worth under 2%.

In [ ]:
t_a = np.nanmean(spec[:, keep], axis=1)      # antenna temperature, K
mins = (t - t[0]) / 60.0

print(f'mean {t_a.mean():.0f} K, rms about a 6th-order trend: '
      f'{100 * (t_a / np.polyval(np.polyfit(mins, t_a, 6), mins) - 1).std():.3f}%')

fig, ax = plt.subplots()
ax.plot(mins, t_a, lw=0.7)
ax.set_xlabel('minutes'); ax.set_ylabel('$T_A$ (K)')
ax.set_title('Band power, uncorrected'); plt.show()

## 6. The tracking scallop

The drive quantises every commanded position to one encoder pulse, **0.5°**.
The firmware rounds, so the pointing error swings ±0.25° rather than running
0 → 0.5, and the beam's gain on a compact source follows the *square* of the
error. The result is a **scallop**, not a sawtooth.

On this run it is the largest systematic left in the photometry. It is also
entirely predictable: the demanded position comes from the ephemeris and the
pointing model, and the quantisation is arithmetic.

**Why this part imports from the repository.** Reconstructing where the mount
was *commanded* to point means reproducing the controller's `trueToDrive` —
refraction, then the pointing-model terms — exactly as the firmware does it. A
hand copy in a notebook would drift away from the firmware silently, so
`scallop.py` uses `drift_park.true_to_drive`, which is validated against
(drive, true) pairs the controller itself reported.

In [ ]:
import scallop          # from receiver_scheduler, put on the path above

ok, why = scallop.applies_to(attrs)
print('scallop correction applies:', ok, why)

terms, source = scallop.pointing_terms(attrs)
print('pointing terms:', source)
print({k: round(v, 3) for k, v in terms.items()})

Recordings made from 2026-09-17 carry the model that was in force as a
`pointing_terms` attribute. An earlier file does not, and falls back to the
model this installation last fitted — an assumption, which is why the source is
reported rather than silently used.

**It must never run with no model at all.** Without one the reconstructed
demand is over a degree out and varies across the sky, the quantisation phase
is wrong everywhere, and the fit returns a *negative* amplitude that reads as
"not detected".

### The error waveform

Computed, not fitted. Note that the altitude steps stretch from about 5 minutes
to half an hour across the run as the Sun's elevation rate falls off — so this
is not a sinusoid and cannot be chased with a periodogram.

In [ ]:
d_alt, d_az, true_alt = scallop.drive_demand(attrs, t, terms)
e_alt, e_az = scallop.sky_offsets(d_alt, d_az, true_alt)

for name, d in (('altitude', d_alt), ('azimuth', d_az)):
    q = np.round(d / scallop.PULSE_DEG) * scallop.PULSE_DEG
    steps = np.flatnonzero(np.diff(q) != 0)
    gaps = np.diff(t[steps]) / 60.0
    print(f'{name:9s} demand {d[0]:7.2f} -> {d[-1]:7.2f} deg, '
          f'{len(steps)+1:3d} pulse steps, '
          f'{gaps.min():.1f}-{gaps.max():.1f} min apart')

fig, ax = plt.subplots()
ax.plot(mins, e_alt, lw=0.8, label='altitude')
ax.plot(mins, e_az, lw=0.8, label='azimuth (projected on sky)')
ax.set_xlabel('minutes'); ax.set_ylabel('pointing error (deg)')
ax.set_title('Where the beam actually sat, relative to the Sun')
ax.legend(fontsize=8); plt.show()

### Fit the depth and the phase

Two things are fitted per axis, and neither is taken from the beam:

- **the amplitude**, because the measured loss runs 10–40% above what a
  Gaussian of the fitted beamwidth predicts — the beam is flat-topped and the
  scallop only ever probes the middle quarter-degree of it;
- **a phase offset**, because the dip sits a few hundredths of a degree from
  where the pointing model puts the pulse boundary. That offset is the model's
  own residual in this patch of sky, and **a correction applied at the wrong
  phase adds modulation instead of removing it.**

Alongside them the fit carries a slow polynomial trend, about one term per half
hour, far too slow to absorb the scallop itself. The beam is still used — as
the plausibility bound that refuses a runaway fit, not as the value.

In [ ]:
fit = scallop.fit(t_a, t, attrs, terms)

print(f"{'':6s} {'k':>9s} {'x beam':>8s} {'sigma':>7s} {'phase':>9s}")
for ax_name in ('alt', 'az'):
    print(f"{ax_name:6s} {fit['k_'+ax_name]:9.4f} "
          f"{fit['k_'+ax_name]/fit['k_beam']:8.2f} "
          f"{fit['sigma_'+ax_name]:7.1f} "
          f"{fit['phase_'+ax_name+'_deg']:+9.4f}")
print()
print('beam', fit['beam_fwhm_deg'], 'deg -> predicted k', round(fit['k_beam'], 4), 'per deg^2')
print('records', fit['records'], '| trend degree', fit['trend_degree'],
      '| axes used', fit['axes_used'])
print('verdict:', fit['why'])

Each axis is judged on its own: below `MIN_SIGMA` (4), or outside 0.25–4× the
beam's predicted curvature, and that axis is left in rather than carried. A
*negative* fitted amplitude applied would amplify the modulation instead of
flattening it, which is the failure this guard exists for.

### Apply it

The gain multiplies the **source**, not the system temperature —
`counts = G·B·(T_sys + g·T_A)` — so the correction belongs here, after T_sys
has been subtracted, and never on the raw counts.

In [ ]:
g = scallop.gain(fit, t, attrs, terms)
t_a_corr = t_a / g

print(f'gain on the source {g.min():.4f} to {g.max():.4f}')
print(f'worst record was {100*(1-g.min()):.2f}% low; mean loss {100*(1-g.mean()):.2f}%')

fig, ax = plt.subplots()
ax.plot(mins, 100 * (g - 1), lw=0.8, color='C1')
ax.set_xlabel('minutes'); ax.set_ylabel('beam gain (%)')
ax.set_title('The correction that gets divided out'); plt.show()

### Did it work?

Fold both series on the drive's own quantisation phase — position within one
encoder pulse — after removing a slow trend. If the scallop is real and the
phase is right, the fold flattens.

In [ ]:
def fold(values, drive, phase=0.0, nbins=12):
    """Peak-to-peak of the de-trended series folded on position within a pulse."""
    resid = values / np.polyval(np.polyfit(mins, values, 8), mins) - 1.0
    ph = np.mod((drive + phase) / scallop.PULSE_DEG, 1.0)
    idx = np.clip((ph * nbins).astype(int), 0, nbins - 1)
    prof = np.array([resid[idx == j].mean() for j in range(nbins)])
    return prof, float(prof.max() - prof.min())


fig, axes = plt.subplots(1, 2, figsize=(11, 3.4))
for ax, (name, drive, phase) in zip(axes, (
        ('altitude', d_alt, fit['phase_alt_deg']),
        ('azimuth', d_az, fit['phase_az_deg']))):
    for series, label, style in ((t_a, 'before', '-o'), (t_a_corr, 'after', '-s')):
        prof, pp = fold(series, drive, phase)
        ax.plot(np.arange(len(prof)) / len(prof), 100 * prof, style, ms=3, lw=1,
                label=f'{label}: {100*pp:.2f}% p-p')
    ax.axhline(0, color='0.8', lw=0.6)
    ax.set_xlabel('position within one 0.5 deg pulse')
    ax.set_ylabel('residual (%)'); ax.set_title(name); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

for label, series in (('before', t_a), ('after', t_a_corr)):
    r = series / np.polyval(np.polyfit(mins, series, 8), mins) - 1.0
    print(f'{label:7s} residual rms {100*r.std():.3f}%  ({r.std()*series.mean():.1f} K)')

## 7. Kelvin to solar flux

The antenna theorem, `A_e = lambda^2 / Omega_A`, with the **measured** main-lobe
solid angle — 23.7 square degrees, integrated directly from three Sun drift
scans, giving 6.18 m² against a physical 7.07.

Treat that as an **upper bound** on the effective area: the sidelobes and the
~10% spillover lie outside the integrated lobe, so the true figure is smaller
by the main-beam efficiency, which has not been measured. The flux is therefore
a slight *under*-estimate.

Then divide by the atmospheric transmission at the Sun's elevation, because a
published index like RSTN's is quoted above the atmosphere. The effect is
small — 1% at the zenith, 3% at airmass 4.

In [ ]:
BOLTZMANN = 1.380649e-23

a_e = float(attrs['effective_area_m2'])
sfu = t_a_corr * 2.0 * BOLTZMANN / (a_e * 1e-22)

# Atmospheric transmission at the Sun's own elevation, record by record.
ZENITH_OPACITY = 0.010                       # nepers, the value the pipeline uses
airmass = 1.0 / np.sin(np.radians(np.clip(true_alt, 3.0, 90.0)))
sfu_top = sfu / np.exp(-ZENITH_OPACITY * airmass)

print(f'effective area {a_e:.2f} m^2 (physical 7.07)')
print(f'Sun altitude {true_alt.min():.1f}-{true_alt.max():.1f} deg, '
      f'transmission {np.exp(-ZENITH_OPACITY*airmass).min():.4f}-'
      f'{np.exp(-ZENITH_OPACITY*airmass).max():.4f}')
print(f'mean flux {sfu_top.mean():.1f} SFU')

## 8. The plot

Binned to about 600 points — the whole run, not its tail — and plotted against
absolute UTC, because solar activity is matched against a flare time or a
published index, never against "minutes in".

In [ ]:
MAX_POINTS = 600
group = max(1, int(np.ceil(sfu_top.size / MAX_POINTS)))
n = (sfu_top.size // group) * group
y = sfu_top[:n].reshape(-1, group).mean(axis=1)
x = t[:n].reshape(-1, group).mean(axis=1)
times = x.astype('datetime64[s]')

fig, ax = plt.subplots(figsize=(11, 4.5))
ax.plot(times, y, lw=1.0, drawstyle='steps-mid')
ax.set_ylabel('Solar flux (SFU, above the atmosphere)')
ax.set_xlabel(f'UTC on {str(times[0])[:10]}')
ax.set_title(f"{attrs.get('obs_name','')}  -  {group} records per point")
fig.autofmt_xdate()
plt.show()

print(f'mean {y.mean():.1f} SFU over {(t[-1]-t[0])/60:.0f} minutes')

### Is that right?

Compare against the reference network. NOAA SWPC publishes the RSTN local-noon
flux at 1415 MHz from Learmonth, San Vito, Sagamore Hill and Palehua. **The
stations disagree with each other by 10–20 SFU on any given day**, so agreement
to within that is agreement, and a single station is not a truth value.

`data/solar_reference_history.json` holds what has been fetched, keyed by date,
so a recording from weeks ago is still captioned with its own day.

In [ ]:
ref = SCHED / 'data' / 'solar_reference_history.json'
if ref.exists():
    hist = json.loads(ref.read_text())
    day = str(np.datetime64(int(t[0]), 's'))[:10]
    entry = hist.get(day) or hist[sorted(hist)[-1]]
    print(f'RSTN 1415 MHz, {day}:')
    print(json.dumps(entry, indent=2)[:600])
else:
    print('no reference history cached yet')

## 9. The one-line version

Everything above is what this call does. Use it for real work; use the notebook
to see what it assumed.

The correction is applied **in the reduction, never written to the file**, so
the recording stays raw and re-reduces the day the pointing model or the
measured beam improves. Nothing here has to be undone.

In [ ]:
import observation_plot

# Written beside this notebook, not into the observatory's data folder.
out = observation_plot.plot_observation(str(h5_path), 'notebook_check.png',
                                        mode='solar')
print('wrote', out)

from IPython.display import Image
Image(out)

## Where to take it

- **Is the beam elliptical?** The fit gives a curvature per axis. On
  2026-09-17 altitude came out 1.20× the Gaussian prediction and azimuth 1.04×,
  a difference of about 2.5 sigma — suggestive, not established. Run this over
  several solar tracks and see whether the split survives.
- **Are the amplitudes a constant of the telescope?** If `k_alt` and `k_az`
  are stable run to run they stop being fitted per observation and become
  measured constants, which would make short runs correctable too.
- **What is left underneath?** After the scallop comes out the residual is
  still around 0.27%, where the radiometer equation plus the measured gain
  instability predict about 0.08%. Something of the same order is unaccounted
  for. The obvious suspect is the rest of the pointing residual, since this
  correction assumes the only error is the quantisation.
- `docs/CALIBRATION.md` has the whole reduction chain: what each step
  measures, what it cannot, and what is still uncalibrated.